## Sentiment Analysis

In this exercise we use the IMDb-dataset, which we will use to perform a sentiment analysis. The code below assumes that the data is placed in the same folder as this notebook. We see that the reviews are loaded as a pandas dataframe, and print the beginning of the first few reviews.

Jeg arbejdede med tekstdata i form af brugeranmeldelser og tilhørende labels (positiv/negativ). For at gøre tekstdata maskinlæsbar, brugte jeg CountVectorizer til at konvertere anmeldelserne til en bag-of-words-repræsentation.
Det betyder, at hver tekst blev til en vektor af ordfrekvenser, som neurale netværk kan træne på – hvor hvert ord er en feature, og værdien er hvor mange gange det forekom i teksten.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
reviews = pd.read_csv('reviews.txt', header=None)
labels = pd.read_csv('labels.txt', header=None)
Y = (labels=='positive').astype(np.int_)

print(type(reviews))
print(reviews.head())

<class 'pandas.core.frame.DataFrame'>
                                                   0
0  bromwell high is a cartoon comedy . it ran at ...
1  story of a man who has unnatural feelings for ...
2  homelessness  or houselessness as george carli...
3  airport    starts as a brand new luxury    pla...
4  brilliant over  acting by lesley ann warren . ...


**(a)** Split the reviews and labels in test, train and validation sets. The train and validation sets will be used to train your model and tune hyperparameters, the test set will be saved for testing. Use the `CountVectorizer` from `sklearn.feature_extraction.text` to create a Bag-of-Words representation of the reviews. Only use the 10,000 most frequent words (use the `max_features`-parameter of `CountVectorizer`).

In [3]:
# Make sure Y is a 1D array (instead of a column).
# Some ML models need the labels in a flat format.
Y = Y.values.ravel()

# Split the data into training+validation (80%) and test set (20%).
# The test set is only used at the very end to check how well the model generalizes.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    reviews[0], Y, test_size=0.2, random_state=42)

# Now split the 80% again into training (60%) and validation (20%).
# The validation set is used to tune and compare models before the final test.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42)

# Create a CountVectorizer (Bag-of-Words) that turns text into numbers.
# We limit it to the 10,000 most frequent words to avoid having too many features.
vectorizer = CountVectorizer(max_features=10000)

# Learn the word counts from the training data only (fit), then transform it.
# This avoids data leakage from validation or test sets.
X_train_bow = vectorizer.fit_transform(X_train)

# Use the same vectorizer to transform validation and test data.
# This keeps the input format consistent across all sets.
X_val_bow = vectorizer.transform(X_val)
X_test_bow = vectorizer.transform(X_test)

# Print out the shape of each dataset.
# This helps check how many samples and features we have at each stage.
print(f"Train set shape: {X_train_bow.shape}")
print(f"Validation set shape: {X_val_bow.shape}")
print(f"Test set shape: {X_test_bow.shape}")


Train set shape: (15000, 10000)
Validation set shape: (5000, 10000)
Test set shape: (5000, 10000)


**(b)** Explore the representation of the reviews. How is a single word represented? How about a whole review?

In [4]:
# Each column in the Bag-of-Words (BoW) matrix stands for a word from the vocabulary.
# Here, we print the first 10 words learned by the vectorizer.
print("First 10 words in the vocabulary:")
print(vectorizer.get_feature_names_out()[:10])

# Pick one example review from the training set to explore.
# This helps us understand how the text is turned into numbers.
review_example = X_train.iloc[0]
print("\nOriginal review text:")
print(review_example)

# Convert the review into a BoW vector.
# The vector shows how many times each known word appears in the review.
# It’s stored in a "sparse" format to save space (most words don’t appear).
review_vector = vectorizer.transform([review_example])
print("\nBoW vector (non-zero entries):")
print(review_vector)

# Now let's find out which words from our vocabulary appear in this review.
# We extract the indices of non-zero values (words that are actually used).
# Then we map those indices back to the actual words.
word_indices = review_vector.indices
words_in_review = [vectorizer.get_feature_names_out()[i] for i in word_indices]

print("\nWords present in the review (according to BoW):")
print(words_in_review)


First 10 words in the vocabulary:
['abandon' 'abandoned' 'abby' 'abc' 'abducted' 'abilities' 'ability'
 'able' 'aboard' 'abominable']

Original review text:
  birth of the beatles   for being a us television movie  released in the fall of     has actually been  so far the best movie which tells the tale of the the four lads from liverpool that revolutionized the music industry and the world . as told by the point of view of former beatle pete best . the performance from the entire cast is excellent but  most especially the performance by stephen mackenna as john lennon and rod culbertson as paul mccartney . the film was produced by a legend of the rock and roll era  mr dick clark . who a year earlier in     had produced another tv movie  that has stood the test of time starring  kurt rusell  in the lead role about another musical legend  elvis  . that movie was directed by an unknown director named  john carpenter  who went on to direct other successful movies such as  halloween    esc

**(c)** Train a neural network with a single hidden layer on the dataset, tuning the relevant hyperparameters to optimize accuracy. 

In [ ]:
# Try out different combinations of hyperparameters for the MLP (neural network).
# We're changing:
# - The size of the hidden layer (number of neurons)
# - The regularization strength (alpha)
# - The learning rate strategy (how learning rate is adjusted during training)
hidden_layer_sizes = [(64,), (128,), (256,)]
alphas = [0.0001, 0.001, 0.01]
learning_rates = ['constant', 'adaptive']

# Store the best model and its validation accuracy
best_model = None
best_val_acc = 0
results = []

# Try all possible combinations of the parameters (grid search)
for size in hidden_layer_sizes:
    for alpha in alphas:
        for lr in learning_rates:
            print(f"Training model: hidden_layer={size}, alpha={alpha}, learning_rate={lr}")
            
            # Create an MLPClassifier (multi-layer perceptron = simple neural network)
            clf = MLPClassifier(hidden_layer_sizes=size,
                                alpha=alpha,
                                learning_rate=lr,
                                max_iter=50,  # How many training iterations (can increase)
                                random_state=42,
                                verbose=False)
            
            # Train the model on the training data
            clf.fit(X_train_bow, y_train)
            
            # Make predictions on the validation set and calculate accuracy
            val_preds = clf.predict(X_val_bow)
            val_acc = accuracy_score(y_val, val_preds)
            results.append((size, alpha, lr, val_acc))
            print(f"Validation Accuracy: {val_acc:.4f}")

            # Keep track of the best-performing model based on validation accuracy
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_model = clf

# Show the best hyperparameter combination found
print("\nBest Model Parameters:")
print(best_model)

# Use the best model to make predictions on the test set
# This gives a final unbiased evaluation of how well the model generalizes
test_preds = best_model.predict(X_test_bow)
test_acc = accuracy_score(y_test, test_preds)
print(f"\nTest Accuracy: {test_acc:.4f}")

Training model: hidden_layer=(64,), alpha=0.0001, learning_rate=constant


**(d)** Test your sentiment-classifier on the test set.

In [9]:
# Use the best model to predict the labels of the test set.
# These are the final predictions used to evaluate how well the model generalizes.
y_test_pred = best_model.predict(X_test_bow)

# Calculate the accuracy on the test set.
# Accuracy = percentage of correct predictions out of all test examples.
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Show a detailed classification report.
# This includes precision, recall, and F1-score for each class.
# These metrics help us understand where the model is doing well or struggling.
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Negative', 'Positive']))

# Show the confusion matrix.
# This table shows how many times each class was correctly or incorrectly predicted.
# It's helpful for spotting specific types of errors (e.g., false positives).
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Test Accuracy: 0.8720

Classification Report:
              precision    recall  f1-score   support

    Negative       0.87      0.87      0.87      2492
    Positive       0.87      0.87      0.87      2508

    accuracy                           0.87      5000
   macro avg       0.87      0.87      0.87      5000
weighted avg       0.87      0.87      0.87      5000

Confusion Matrix:
[[2174  318]
 [ 322 2186]]


**(e)** Use the classifier to classify a few sentences you write yourselves. 

In [10]:
# These are some example sentences we want to classify.
# This simulates how the model might be used in real-world applications (e.g., user reviews).
custom_sentences = [
    "I absolutely loved this movie! It was fantastic.",
    "What a waste of time. The plot was boring and predictable.",
    "It was okay, not great but not terrible either.",
    "The acting was brilliant, but the story made no sense.",
    "Terrible. I walked out halfway through."
]

# Convert the new sentences into Bag-of-Words format using the same vectorizer as before.
# This step turns raw text into numerical input that the model can understand.
custom_bow = vectorizer.transform(custom_sentences)

# Predict the sentiment (Positive or Negative) using the trained model.
# This is the classification step.
predictions = best_model.predict(custom_bow)

# Convert model output (0 or 1) back to readable sentiment labels.
# Then print each sentence along with its predicted sentiment.
labels = ['Negative', 'Positive']
for sentence, pred in zip(custom_sentences, predictions):
    print(f"Sentence: {sentence}\nPredicted Sentiment: {labels[pred]}\n")

Sentence: I absolutely loved this movie! It was fantastic.
Predicted Sentiment: Positive

Sentence: What a waste of time. The plot was boring and predictable.
Predicted Sentiment: Negative

Sentence: It was okay, not great but not terrible either.
Predicted Sentiment: Negative

Sentence: The acting was brilliant, but the story made no sense.
Predicted Sentiment: Positive

Sentence: Terrible. I walked out halfway through.
Predicted Sentiment: Negative

